# evaluatorB_frame â€” Frame-level scorer (binary INTERACTING / NOT INTERACTING)

Frame counterpart of `evaluatorB.ipynb`. Run all cells top-to-bottom.

**Protocol** (binary frame decision): each stride-grid frame is labeled INTERACTING if any interval covers it, else NOT INTERACTING. Confusion matrix (TP/FP/FN/TN) â†’ Precision, Recall, F1. TN is reported for the matrix; accuracy is excluded from headlines (TN-dominated).

**Pred source**: stitched summary intervals (`logs_summary`, same input as the interval metric).

**ID diagnostic**: `(frame, human_id, object_id)` triple-set match (idP/idR/idF1 + gap vs binary F1). Diagnostic only — isolates frame-level identity cost; never a headline.

Universe per video: stride-5 grid `0 â€¦ max_end`, `max_end` = max over GT ends and summary ends. Inclusive ends (`s â‰¤ f â‰¤ e`). P/R/F1 are invariant to the cap (TN-only tail).

In [ ]:
from pathlib import Path

import pandas as pd

CWD = Path.cwd()
if (CWD / "eval" / "gt_summary").exists():
    ROOT = CWD            # kernel launched from src/
elif (CWD / "gt_summary").exists():
    ROOT = CWD.parent     # kernel launched from src/eval/
else:
    raise SystemExit(f"Cannot locate gt_summary from {CWD}")

GT_DIR = ROOT / "eval" / "gt_summary"
SUM_DIR = ROOT / "video" / "logs" / "logs_summary"
OUT_CSV = ROOT / "eval" / "result" / "primary_frames.csv"
STRIDE = 5
GT_PATTERN = "vid*_true_summary.csv"
SUM_PATTERN = "vid*_summary_log.csv"

pd.options.display.float_format = "{:.2f}".format

print(f"GT:   {GT_DIR}")
print(f"SUM:  {SUM_DIR}")
print(f"OUT:  {OUT_CSV}")
print(f"STRIDE: {STRIDE}")

In [ ]:
import re


def _vidnum(name):
    m = re.search(r"vid(\d+)", name)
    return int(m.group(1)) if m else None


def expand_intervals(intervals, max_end, stride):
    """Stride-grid frames covered by any inclusive [start, end] interval."""
    active = set()
    for s, e in intervals:
        first = s + ((stride - s % stride) % stride)  # first multiple of stride >= s
        for f in range(first, e + 1, stride):
            active.add(f)
    return active


def expand_triples(df, stride):
    """(frame, human_id, object_id) triples on the stride grid."""
    out = set()
    for r in df.itertuples():
        s, e = int(r.frame_start), int(r.frame_end)
        first = s + ((stride - s % stride) % stride)
        for f in range(first, e + 1, stride):
            out.add((f, int(r.human_id), int(r.object_id)))
    return out


def score_triples(gt_triples, pred_triples):
    tp = len(gt_triples & pred_triples)
    fp = len(pred_triples - gt_triples)
    fn = len(gt_triples - pred_triples)
    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    return {"idTP": tp, "idFP": fp, "idFN": fn, "idP": P, "idR": R, "idF1": F}


def score_frames(grid, gt_active, pred_active):
    tp = len(gt_active & pred_active)
    fp = len(pred_active - gt_active)
    fn = len(gt_active - pred_active)
    tn = len(set(grid) - gt_active - pred_active)
    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn, "P": P, "R": R, "F1": F}

In [ ]:
gt = {n: f for f in GT_DIR.glob(GT_PATTERN) if (n := _vidnum(f.name)) is not None}
sm = {n: f for f in SUM_DIR.glob(SUM_PATTERN) if (n := _vidnum(f.name)) is not None}
pairs = [(v, gt[v], sm[v]) for v in sorted(set(gt) & set(sm))]
for v in sorted(set(gt) - set(sm)):
    print(f"skip vid{v:02d}: GT without prediction ({gt[v].name})")
for v in sorted(set(sm) - set(gt)):
    print(f"skip vid{v:02d}: prediction without GT ({sm[v].name})")
print(f"scoring {len(pairs)} videos")

In [ ]:
rows, id_rows = [], []
for v, gp, sp in pairs:
    gdf = pd.read_csv(gp)
    sdf = pd.read_csv(sp)
    g_iv = gdf[["frame_start", "frame_end"]].values.tolist()
    s_iv = sdf[["frame_start", "frame_end"]].values.tolist()
    max_end = max(e for _, e in g_iv + s_iv)
    grid = list(range(0, max_end + 1, STRIDE))
    gt_active = expand_intervals(g_iv, max_end, STRIDE)
    s_active = expand_intervals(s_iv, max_end, STRIDE)
    rows.append({"video": f"vid{v:02d}", "n_frames": len(grid), **score_frames(grid, gt_active, s_active)})
    t = score_triples(expand_triples(gdf, STRIDE), expand_triples(sdf, STRIDE))
    id_rows.append({"video": f"vid{v:02d}", **t})
df = pd.DataFrame(rows)
df_id = pd.DataFrame(id_rows)
df_id["gap"] = df.set_index("video").loc[df_id["video"], "F1"].to_numpy() - df_id["idF1"].to_numpy()
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
df


In [ ]:
print(f"=== DATASET (mean +- sample std, n={len(df)}) ===")
for c in ("P", "R", "F1"):
    print(f"{c:3s} {df[c].mean():.2f} +- {df[c].std(ddof=1):.2f}")
print()
print("=== ID-TRIPLE diagnostic ===")
for c in ("idP", "idR", "idF1"):
    print(f"{c:4s} {df_id[c].mean():.2f} +- {df_id[c].std(ddof=1):.2f}")
print(f"mean identity gap (F1 - idF1): {df_id['gap'].mean():+.2f}")
print(f"gap >= 0.30: {df_id.loc[df_id.gap >= 0.30, 'video'].tolist()}")

df = df.merge(df_id[["video", "idTP", "idFP", "idFN", "idP", "idR", "idF1", "gap"]], on="video")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}")
df_id
